In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
# -*- coding: utf-8 -*-
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
import ast, re

# -------------------------------
# Utilities
# -------------------------------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

def try_get_widget(name, default=None):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

def coerce_list(x, default=None):
    if default is None: default = []
    if isinstance(x, list): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else default
        except Exception:
            return default
    return default

def coerce_dict(x, default=None):
    if default is None: default = {}
    if isinstance(x, dict): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else default
        except Exception:
            return default
    return default

# --- CTL helpers ---
def read_ctl_lines(path, sample_only=False, sample_limit=10000):
    if not path:
        return []
    try:
        df = spark.read.text(path)
        rows = df.collect() if not sample_only else df.limit(sample_limit).collect()
        return [r['value'] for r in rows]
    except Exception:
        try:
            content = dbutils.fs.head(path, 1024 * 1024)
            return content.splitlines()
        except Exception:
            return []

def parse_ctl_bar_format(lines):
    names = []
    for row in lines or []:
        parts = row.split('|')
        if len(parts) >= 1:
            fn = parts[0].strip()
            if fn.endswith('.gz') and fn:
                names.append(fn)
    # de-dupe, preserve order
    seen = set(); out = []
    for n in names:
        if n not in seen:
            seen.add(n); out.append(n)
    return out

def base_name(fn):
    """Extract base name without '.YYYYMMDD.gz' or '.gz'."""
    m = re.match(r'^(.*?)(\.\d{8})?\.gz$', fn)
    return m.group(1) if m else fn

def detect_date_suffix_from_names(names):
    """Capture '.YYYYMMDD.gz' from any CTL file name to reconstruct full names in messages."""
    for n in names:
        m = re.search(r'\.(\d{8})\.gz$', n)
        if m:
            return f".{m.group(1)}.gz"
    return ".gz"



# -------------------------------
# Inputs (from Pre_validation + widgets)
# -------------------------------
error_table = try_get_widget("error_table")
ctl_path    = try_get_widget("ctl_path") or safe_get("Pre_validation", "ctl_path", None)

ctl_data = []
if ctl_path:
    ctl_df = spark.read.text(ctl_path)
    ctl_data = ctl_df.collect()

ctl_gz_basenames = set()  # default empty set
for row in ctl_data:
    parts = row[0].split('|')
    if len(parts) >= 2:
        file_name = parts[0].strip()
        if should_skip(file_name):
            continue
        match = re.match(r"^(.*?)(\.\d{8})?\.gz$", file_name)
        if match:
            base_name = match.group(1)
            ctl_gz_basenames.add(base_name)

# Task values from Pre_validation
missing_required_raw     = safe_get("Pre_validation", "missing_required_gz_files", [])
gz_file_paths_raw        = safe_get("Pre_validation", "gz_file_paths", {})
gz_expected_info_raw     = safe_get("Pre_validation", "gz_expected_info", [])
ctl_expected_files_raw   = safe_get("Pre_validation", "ctl_expected_files", [])
required_gz_list         = safe_get("Pre_validation", "required_gz_list", [])

# Timestamp keys (prefer your existing ctl_received_ts; fallback to ctl_run_ts)
ctl_run_ts               = safe_get("Pre_validation", "ctl_run_ts", None)
start_load = safe_get("Pre_validation", "start_time", None)

# Coerce
missing_required_gz_files = coerce_list(missing_required_raw, default=[])
gz_file_paths             = coerce_dict(gz_file_paths_raw, default={})
ctl_expected_files        = coerce_list(ctl_expected_files_raw, default=[])

missing_required_files = [
    name for name in required_gz_list if name not in ctl_gz_basenames
]
# Timestamps
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Determine per-row Date_Received default for CTL-present files

ctl_present_default_ts = ctl_run_ts if ctl_run_ts else datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Build CTL expected list (full names)
expected_names = []
if ctl_expected_files:
    expected_names = [n for n in ctl_expected_files if isinstance(n, str) and n.endswith('.gz')]
elif ctl_path:
    lines = read_ctl_lines(ctl_path)
    expected_names = parse_ctl_bar_format(lines)

# If CTL list still empty and CTL passed (gz_expected_info available), flatten it
if not expected_names and gz_expected_info_raw:
    gzei = coerce_list(gz_expected_info_raw, default=[])
    for item in gzei:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            expected_names.append(item[0])
        elif isinstance(item, dict) and "file_name" in item:
            expected_names.ap4pend(item["file_name"])
        elif isinstance(item, str):
            expected_names.append(item)

# De-dupe, preserve order
seen = set(); expected_names = [n for n in expected_names if (n not in seen and not seen.add(n))]

missing_required_base_set = set(missing_required_gz_files)

# -------------------------------
# Build error records
# -------------------------------
error_records = []
phase = None

error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

if missing_required_gz_files:
    # ---------- CTL phase ----------
    phase = "CTL"

    # Reconstruct full names for error message using suffix from CTL list
    suffix = detect_date_suffix_from_names(expected_names)
    missing_full_names = [f"{b}{suffix}" for b in missing_required_gz_files]
    common_err = (
        f"MISSING {missing_full_names[0]} FILE."
        if len(missing_full_names) == 1
        else "MISSING " + ", ".join(missing_full_names) + " FILES."
    )

    # 1. Rows for files listed in CTL
    for file_name in expected_names:
        bn = base_name(file_name)
        date_received = "" if bn in missing_required_base_set else ctl_present_default_ts
        error_records.append((file_name, date_received, start_load, end_load, 0, common_err))

    # 2. Extra rows for files missing from CTL
    for bn in missing_required_gz_files:
        file_name = f"{bn}{suffix}"
        error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

else:
    # ---------- S3 phase ----------
    phase = "S3"
    # expected_names for S3 should come from gz_expected_info when CTL passed
    if not expected_names and gz_expected_info_raw:
        gzei = coerce_list(gz_expected_info_raw, default=[])
        for item in gzei:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                expected_names.append(item[0])
            elif isinstance(item, dict) and "file_name" in item:
                expected_names.append(item["file_name"])
            elif isinstance(item, str):
                expected_names.append(item)
        # de-dupe
        seen = set(); expected_names = [n for n in expected_names if (n not in seen and not seen.add(n))]

    keys = set(gz_file_paths.keys())
    missing_in_s3 = [name for name in expected_names if name not in keys]
    for file_name in missing_in_s3:
        error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

# -------------------------------
# Write
# -------------------------------
if error_records:
    error_df = spark.createDataFrame(error_records, schema=error_schema)
    error_df.write.mode("append").saveAsTable(error_table)
    print(f"✅ Error report written to {error_table} (phase={phase}, count={len(error_records)})")
    display(error_df)
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=True)
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=False)

# -------------------------------
# Debug
# -------------------------------
print(f"🔹 Phase: {phase}")
print(f"🔹 ctl_path: {ctl_path}")
print(f"🔹 Start Load: {start_load}")
print(f"🔹 End Load: {end_load}")
print(f"🔹 ctl_run_ts (fallback): {ctl_run_ts}")
print(f"🔹 required_Gz_files count: {len(required_gz_list)}")
print(f"🔹 received_files (from CTL) count: {len(ctl_gz_basenames)}")
print(f"🔹 missing_files (required not in CTL): {len(missing_required_files)} → {missing_required_files}")
print(f"🔹 Total Error Records: {len(error_records)}")
